# Lahore Road Network (OpenStreetMap)
## Export: filtered road network GeoPackage


### 0. Initialize Imports


In [1]:
import geopandas as gpd
import osmnx as ox


/Users/ahmed/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### 1. Parameters


In [2]:
BOUNDARY_PATH = "../lahore.geojson"
TARGET_CRS = "EPSG:32643"
OUT_GPKG = "lahore_roads.gpkg"
OUT_LAYER = "roads"

HIGHWAY_PATTERN = "motorway|trunk|primary|secondary|tertiary|unclassified|residential|service|living_street"


### 2. Load Lahore Boundary and Fetch OSM Roads


In [3]:
boundary = gpd.read_file(BOUNDARY_PATH)
if boundary.crs is None or boundary.crs.to_epsg() != 4326:
    boundary = boundary.to_crs(4326)
lahore_boundary = boundary.geometry.unary_union

if hasattr(ox, "features_from_polygon"):
    roads = ox.features_from_polygon(lahore_boundary, tags={"highway": True})
else:
    roads = ox.geometries_from_polygon(lahore_boundary, tags={"highway": True})

roads = roads[roads.geometry.type.isin(["LineString", "MultiLineString"])].copy()
if "highway" in roads.columns:
    roads = roads[
        roads["highway"].astype(str).str.contains(
            HIGHWAY_PATTERN,
            regex=True,
            na=False,
        )
    ].copy()

roads = roads.to_crs(TARGET_CRS)
print(f"Road segments fetched: {len(roads)}")


/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_41339/1106564760.py:4: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  lahore_boundary = boundary.geometry.unary_union


Road segments fetched: 96495


### 3. Export GeoPackage


In [4]:
roads.to_file(OUT_GPKG, layer=OUT_LAYER, driver="GPKG")
print(f"Saved: {OUT_GPKG} ({OUT_LAYER})")


Saved: lahore_roads.gpkg (roads)
